# 02b — Groundsource: Pluvial Flood Analysis

**Purpose:** Visualise and interpret the Pluvial Flood Detection Index (PFDI) to understand
what fraction of flood events are pluvial (local rainfall) versus fluvial (river-driven).
PFDI < 1 indicates the upstream drainage area is smaller than the flood polygon — a signature
of localised, pluvial flooding.

**Input:**
- `outputs/groundsource_df_with_pfdi.parquet` — flood events with PFDI metrics (from 02a)

**Output:**
- Charts and printed summaries only. No files are saved.

In [ ]:
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

warnings.filterwarnings('ignore')

In [ ]:
# --- CONFIGURATION ---

INPUT_PATH     = r"D:\MY_CODES\global_urban_flood_database\groundsource\outputs\groundsource_df_with_pfdi.parquet"
AREA_THRESHOLD = 500   # km² — focus analysis on smaller, pluvial-prone polygons
AREA_BINS      = [0, 10, 50, 100, 250, 500]   # km² bins for stratified analysis
AREA_LABELS    = ['< 10', '10–50', '50–100', '100–250', '250–500']

## 1. Load and validate

In [ ]:
df = pd.read_parquet(INPUT_PATH)
print(f"Loaded {len(df):,} records")
print(df[['upa_p95', 'upa_p99', 'upa_max', 'poly_area_km2',
          'PFDI_p95', 'PFDI_p99', 'PFDI_max']].describe())

# Overall pluvial fractions (PFDI ≤ 1 = pluvial)
for col in ['PFDI_p95', 'PFDI_p99', 'PFDI_max']:
    pct = (df[col] <= 1.0).mean() * 100
    print(f"  {col} ≤ 1 (pluvial): {pct:.1f}%")

## 2. Log-log scatter: polygon area vs upstream drainage area

In [ ]:
sample = df.sample(n=min(20_000, len(df)), random_state=42)

fig, ax = plt.subplots(figsize=(9, 7))
sc = ax.scatter(
    sample['poly_area_km2'].clip(lower=1e-3),
    sample['upa_p99'].clip(lower=1e-3),
    s=3, alpha=0.3, c=np.log10(sample['PFDI_max'].clip(lower=1e-4)),
    cmap='RdYlBu', vmin=-2, vmax=2
)
plt.colorbar(sc, ax=ax, label='log₁₀(PFDI_max)  [blue = fluvial, red = pluvial]')

# Identity line: UPA = area → PFDI = 1
lim = [1e-3, 1e5]
ax.plot(lim, lim, 'k--', linewidth=1, label='UPA = Area (PFDI = 1)')
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlim(lim)
ax.set_ylim(lim)
ax.set_xlabel('Polygon Area (km²)')
ax.set_ylabel('UPA p99 (km²)')
ax.set_title('Polygon Area vs Upstream Drainage Area — Groundsource')
ax.legend()
plt.tight_layout()
plt.show()

## 3. PFDI distribution by area bin (≤ 500 km²)

In [ ]:
small = df[df['poly_area_km2'] <= AREA_THRESHOLD].copy()
small['area_bin'] = pd.cut(small['poly_area_km2'], bins=AREA_BINS, labels=AREA_LABELS)

print(f"Events ≤ {AREA_THRESHOLD} km²: {len(small):,}")
print("\nPluvial fraction (PFDI_max ≤ 1) by area bin:")
for label, grp in small.groupby('area_bin', observed=True):
    pct = (grp['PFDI_max'] <= 1.0).mean() * 100
    print(f"  {label:>10} km²  |  n={len(grp):>7,}  |  pluvial: {pct:.1f}%")

In [ ]:
# Box plots of PFDI_max by area bin
groups = [small[small['area_bin'] == lbl]['PFDI_max'].clip(1e-4, 1e4)
          for lbl in AREA_LABELS if lbl in small['area_bin'].values]

fig, ax = plt.subplots(figsize=(10, 6))
ax.boxplot(groups, labels=AREA_LABELS, showfliers=False)
ax.axhline(1.0, color='red', linestyle='--', label='PFDI = 1 (pluvial / fluvial boundary)')
ax.set_yscale('log')
ax.set_xlabel('Polygon Area Bin (km²)')
ax.set_ylabel('PFDI_max (log scale)')
ax.set_title('PFDI Distribution by Area Bin — Groundsource')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Density hexbin: area vs PFDI

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
hb = ax.hexbin(
    np.log10(small['poly_area_km2'].clip(lower=1e-3)),
    np.log10(small['PFDI_max'].clip(lower=1e-4, upper=1e4)),
    gridsize=40, cmap='inferno_r', mincnt=1
)
plt.colorbar(hb, ax=ax, label='Event count per hex bin')
ax.axhline(0, color='cyan', linestyle='--', linewidth=1.2,
           label='PFDI = 1 (boundary)')
ax.set_xlabel('log₁₀(Polygon Area [km²])')
ax.set_ylabel('log₁₀(PFDI_max)')
ax.set_title('Density: Polygon Area vs PFDI_max — Groundsource (≤ 500 km²)')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Summary

In [ ]:
print("=" * 55)
print("  PFDI ANALYSIS SUMMARY")
print("=" * 55)
for col in ['PFDI_p95', 'PFDI_p99', 'PFDI_max']:
    pct = (df[col] <= 1.0).mean() * 100
    print(f"  {col} ≤ 1 (pluvial) : {pct:>6.1f}%")
print(f"  Median PFDI_max      : {df['PFDI_max'].median():>10.3f}")
print(f"  Median poly_area_km2 : {df['poly_area_km2'].median():>10.3f} km²")
print("=" * 55)